# Posterior HDF5 recover test

Discover/filter `(decoded_posteriors)_tbin-*.h5` like the paper archive notebook, load arrays via `LoadedPosteriorContainer`, then recover original `SingleEpochDecodedResult` objects with `read_hdf`.


In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# required for PhoPaginatedMultiDecoderDecodedEpochsWindow / Qt widgets:
%gui qt5

from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Any
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import h5py

pd.set_option('display.max_columns', 100)

from neuropy.core.user_annotations import UserAnnotationsManager
from neuropy.utils.result_context import IdentifyingContext
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path
from pyphocorehelpers.Filesystem.HDF5.hdf5_file_helpers import HDF5_Helper
from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import load_across_sessions_exported_h5_files, get_only_most_recent_session_files, find_csv_files, find_most_recent_files, AcrossSessionHelpers
from pyphoplacecellanalysis.Pho2D.data_exporting import PosteriorExporting, LoadedPosteriorContainer
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import SingleEpochDecodedResult, DecodedFilterEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.DecoderPredictionError import OverlayLabelsPlotData
from neuropy.core.epoch import ensure_dataframe


## Resolve `collected_outputs_directory`


In [ ]:
known_collected_outputs_paths = [Path(v).resolve() for v in ['/Users/pho/data/collected_outputs',
                                                            '/Volumes/SwapSSD/Data/collected_outputs', r"K:/scratch/collected_outputs", '/Users/pho/Dropbox (University of Michigan)/MED-DibaLabDropbox/Data/Pho/Outputs/output/collected_outputs', r'C:/Users/pho/repos/Spike3DWorkEnv/Spike3D/output/collected_outputs',
                                                            '/home/halechr/cloud/turbo/Data/Output/collected_outputs',
                                                            '/home/halechr/cloud/turbo/Pho/Output/collected_outputs',
                                                            '/home/halechr/FastData/collected_outputs/',
                                                            ]]
collected_outputs_directory = find_first_extant_path(known_collected_outputs_paths)
assert collected_outputs_directory.exists(), f"collected_outputs_directory: {collected_outputs_directory} does not exist! Is the right computer's config commented out above?"
print(f'collected_outputs_directory: {collected_outputs_directory}')

known_bad_sessions: List[IdentifyingContext] = UserAnnotationsManager.get_hardcoded_bad_sessions()
known_bad_session_strs = [str(v.get_description()) for v in known_bad_sessions]


## Index exported H5 files


In [ ]:
cuttoff_date = datetime(2026, 11, 2)
print(f'cuttoff_date: {cuttoff_date}')

parsed_h5_files_df, h5_contexts_paths_dict = load_across_sessions_exported_h5_files(collected_outputs_directory=collected_outputs_directory, cuttoff_date=cuttoff_date,
                                                                                    known_bad_session_strs=known_bad_session_strs, debug_print=True)
parsed_h5_files_df['file_type'].unique()
parsed_h5_files_df


## Filter to `decoded_posteriors` (archive parity)


In [ ]:
## INPUTS: parsed_h5_files_df
decoded_posteriors_parsed_h5_files_df = parsed_h5_files_df[parsed_h5_files_df['file_type'] == 'decoded_posteriors']

print(decoded_posteriors_parsed_h5_files_df['custom_replay_name'].unique())
# matching_custom_replay_name_str: str = "withNormalComputedReplays-frateThresh_5.0-qclu_[1, 2]"
# matching_custom_replay_name_str: str = "withNormalComputedReplays-frateThresh_5.0-qclu_[1, 2, 4, 6, 7, 9]"
# matching_custom_replay_name_str: str = 'withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 9]-frateThresh_5.0'
# matching_custom_replay_name_str: str = 'withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 8, 9]-frateThresh_2.0'
matching_custom_replay_name_str: str = 'withNormalComputedReplays-qclu-[1, 2, 4, 6, 7, 8, 9]-frateThresh-2.0'
# compare_custom_replay_name_col_name: str = 'custom_replay_name'
compare_custom_replay_name_col_name: str = '_comparable_custom_replay_name'

decoding_time_bin_size_str: str = '0.025'
# decoding_time_bin_size_str: str = '0.075'

assert compare_custom_replay_name_col_name in decoded_posteriors_parsed_h5_files_df.columns
decoded_posteriors_parsed_h5_files_df = decoded_posteriors_parsed_h5_files_df.pho.constrain_df_cols(decoding_time_bin_size_str=decoding_time_bin_size_str, **{compare_custom_replay_name_col_name:matching_custom_replay_name_str}, should_drop_constrained_columns=False)
decoded_posteriors_parsed_h5_files_df = decoded_posteriors_parsed_h5_files_df.sort_values(['session', 'file_type', compare_custom_replay_name_col_name, 'decoding_time_bin_size_str', 'export_datetime']).reset_index(drop=True)
decoded_posteriors_parsed_h5_files_df = get_only_most_recent_session_files(parsed_paths_df=deepcopy(decoded_posteriors_parsed_h5_files_df), group_column_names=['session', compare_custom_replay_name_col_name, 'file_type', 'decoding_time_bin_size_str'])
decoded_posteriors_h5_files = [Path(v.as_posix()).resolve() for v in decoded_posteriors_parsed_h5_files_df['path'].to_list()]
decoded_posteriors_parsed_h5_files_df
## OUTPUTS: decoded_posteriors_h5_files, decoded_posteriors_parsed_h5_files_df


In [ ]:
decoded_posteriors_h5_files


## Array load (existing `LoadedPosteriorContainer` path)


In [ ]:
all_sessions_exported_posteriors_dict, all_sessions_exported_posteriors_data_only_dict = LoadedPosteriorContainer.load_batch_hdf5_exports(exported_posterior_data_h5_files=decoded_posteriors_h5_files)
## OUTPUTS: all_sessions_exported_posteriors_dict, all_sessions_exported_posteriors_data_only_dict
list(all_sessions_exported_posteriors_dict.keys())


## Recover original `SingleEpochDecodedResult` objects via `read_hdf`

Batch `(decoded_posteriors).h5` layout is per-epoch under `…/save_decoded_posteriors_to_HDF5/{decoder}/{laps|ripple}/{epoch_idx}/` — not the `DecodedFilterEpochsResult.read_hdf` `{key}/epochs/{i}` layout.


In [ ]:
def recover_single_epoch_decoded_results_from_hdf5(load_path: Path, debug_print: bool=False) -> Tuple[Dict[str, Dict[str, Dict[str, SingleEpochDecodedResult]]], str]:
    """Walk PosteriorExporting batch layout and restore each epoch with SingleEpochDecodedResult.read_hdf.

    Returns:
        restored_by_decoder_epochs: Dict[decoder_name, Dict[epochs_name, Dict[epoch_idx_str, SingleEpochDecodedResult]]]
        save_key: HDF5 group path ending in 'save_decoded_posteriors_to_HDF5'
    """
    load_path = Path(load_path).resolve()
    assert load_path.exists(), f"load_path: '{load_path}' does not exist!"

    found_groups = HDF5_Helper.find_groups_by_name(load_path, 'save_decoded_posteriors_to_HDF5')
    assert len(found_groups) == 1, f"expected exactly one save_decoded_posteriors_to_HDF5 group, got: {found_groups}"
    save_key: str = found_groups[0]
    if debug_print:
        print(f'save_key: {save_key}')

    restored_by_decoder_epochs: Dict[str, Dict[str, Dict[str, SingleEpochDecodedResult]]] = {}
    with h5py.File(load_path, 'r') as f:
        main_save_group = f[save_key]
        for decoder_name in main_save_group.keys():
            restored_by_decoder_epochs[decoder_name] = {}
            decoder_group = main_save_group[decoder_name]
            for epochs_name in decoder_group.keys():
                restored_by_decoder_epochs[decoder_name][epochs_name] = {}
                epochs_group = decoder_group[epochs_name]
                for epoch_idx_str in sorted(epochs_group.keys(), key=lambda s: int(s) if str(s).isdigit() else str(s)):
                    epoch_group_key: str = f'{save_key}/{decoder_name}/{epochs_name}/{epoch_idx_str}'
                    restored = SingleEpochDecodedResult.read_hdf(f, key=epoch_group_key)
                    restored_by_decoder_epochs[decoder_name][epochs_name][epoch_idx_str] = restored
                ## END for epoch_idx_str in sorted(epochs_group.keys(), key=lambda s: int(s) if str(s).isdigit() else str(s))...

            ## END for epochs_name in decoder_group.keys()...

        ## END for decoder_name in main_save_group.keys()...

    ## END with h5py.File(load_path, 'r') as f...

    return restored_by_decoder_epochs, save_key


## Recover from the first filtered file (extend to all paths as needed):
assert len(decoded_posteriors_h5_files) > 0, "no decoded_posteriors h5 files after filtering"
active_posterior_h5_path: Path = decoded_posteriors_h5_files[0]
print(f'active_posterior_h5_path: {active_posterior_h5_path}')

restored_by_decoder_epochs, active_save_key = recover_single_epoch_decoded_results_from_hdf5(active_posterior_h5_path, debug_print=True)
## OUTPUTS: restored_by_decoder_epochs, active_save_key, active_posterior_h5_path
{decoder: {epochs_name: len(epoch_dict) for epochs_name, epoch_dict in epochs_dict.items()} for decoder, epochs_dict in restored_by_decoder_epochs.items()}


## Smoke checks — object fields + spot-compare vs array loader


In [ ]:
## Pick first available decoder / epochs_name / epoch:
example_decoder_name: str = next(iter(restored_by_decoder_epochs.keys()))
example_epochs_name: str = next(iter(restored_by_decoder_epochs[example_decoder_name].keys()))
example_epoch_idx_str: str = next(iter(restored_by_decoder_epochs[example_decoder_name][example_epochs_name].keys()))
example_epoch_idx: int = int(example_epoch_idx_str)
restored: SingleEpochDecodedResult = restored_by_decoder_epochs[example_decoder_name][example_epochs_name][example_epoch_idx_str]

print(f'example: decoder={example_decoder_name!r}, epochs={example_epochs_name!r}, epoch_idx={example_epoch_idx_str!r}')
print(f'type(restored): {type(restored)}')
print(f'restored.nbins: {restored.nbins}')
print(f'restored.epoch_data_index: {restored.epoch_data_index}')
print(f'restored.p_x_given_n.shape: {np.shape(restored.p_x_given_n)}')
print(f'restored.most_likely_positions.shape: {np.shape(restored.most_likely_positions) if restored.most_likely_positions is not None else None}')

example_epoch_group_key: str = f'{active_save_key}/{example_decoder_name}/{example_epochs_name}/{example_epoch_idx_str}'
with h5py.File(active_posterior_h5_path, 'r') as f:
    example_group = f[example_epoch_group_key]
    has_p_x_given_n_full: bool = ('p_x_given_n_full' in example_group)
    save_full_object_attr = example_group.attrs.get('save_full_object', None)
    print(f'has_p_x_given_n_full: {has_p_x_given_n_full}')
    print(f'save_full_object attr: {save_full_object_attr}')
    legacy_p_x_given_n = np.asarray(example_group['p_x_given_n'])

## Array-loader comparison (legacy dataset `p_x_given_n` is the x-marginal):
array_out_dict, _meta = PosteriorExporting.load_decoded_posteriors_from_HDF5(load_path=active_posterior_h5_path, debug_print=False)
array_p_x_given_n = array_out_dict[example_decoder_name][example_epochs_name]['p_x_given_n'][example_epoch_idx]
print(f'array_p_x_given_n.shape: {np.shape(array_p_x_given_n)}')
np.testing.assert_allclose(legacy_p_x_given_n, array_p_x_given_n)

## If full object was saved, restored.p_x_given_n may be higher-D than the legacy x-marginal:
if has_p_x_given_n_full:
    if restored.marginal_x is not None:
        np.testing.assert_allclose(restored.marginal_x.p_x_given_n, array_p_x_given_n)
        print('OK: restored.marginal_x.p_x_given_n matches array loader x-marginal')
    else:
        print('note: restored.marginal_x is None; skipped marginal compare')
else:
    np.testing.assert_allclose(restored.p_x_given_n, array_p_x_given_n)
    print('OK: legacy file — restored.p_x_given_n matches array loader (x-marginal)')

restored


## Reassemble approximate `DecodedFilterEpochsResult` containers

Batch PosteriorExporting files do not use `DecodedFilterEpochsResult.to_hdf` layout, so wrap each recovered `SingleEpochDecodedResult` via `init_from_single_epoch_result` and merge with `init_by_merging_multiple_epoch_results`. `decoding_time_bin_size` comes from the filename `_tbin-…` (container-level attrs / full `filter_epochs` / `pos_bin_edges` may be incomplete vs the original export).

In [ ]:
## INPUTS: restored_by_decoder_epochs, active_posterior_h5_path
decoding_time_bin_size: Optional[float] = LoadedPosteriorContainer.parse_time_bin_size_from_path(active_posterior_h5_path)
assert decoding_time_bin_size is not None, f"could not parse `_tbin-…` from filename: {active_posterior_h5_path.name}"
print(f'decoding_time_bin_size: {decoding_time_bin_size}')

restored_decoded_filter_epochs_results_dict: Dict[str, Dict[str, DecodedFilterEpochsResult]] = {}
for decoder_name, epochs_dict in restored_by_decoder_epochs.items():
    restored_decoded_filter_epochs_results_dict[decoder_name] = {}
    for epochs_name, epoch_idx_to_result in epochs_dict.items():
        single_epoch_containers: List[DecodedFilterEpochsResult] = []
        for epoch_idx_str in sorted(epoch_idx_to_result.keys(), key=lambda s: int(s) if str(s).isdigit() else str(s)):
            single_epoch_containers.append(DecodedFilterEpochsResult.init_from_single_epoch_result(epoch_idx_to_result[epoch_idx_str], decoding_time_bin_size=float(decoding_time_bin_size)))
        ## END for epoch_idx_str in sorted(epoch_idx_to_result.keys(), key=lambda s: int(s) if str(s).isdigit() else str(s))...

        if len(single_epoch_containers) == 0:
            continue
        restored_decoded_filter_epochs_results_dict[decoder_name][epochs_name] = DecodedFilterEpochsResult.init_by_merging_multiple_epoch_results(single_epoch_containers)
    ## END for epochs_name, epoch_idx_to_result in epochs_dict.items()...

## END for decoder_name, epochs_dict in restored_by_decoder_epochs.items()...

## OUTPUTS: restored_decoded_filter_epochs_results_dict
{decoder: {epochs_name: (v.num_filter_epochs, float(v.decoding_time_bin_size)) for epochs_name, v in epochs_dict.items()} for decoder, epochs_dict in restored_decoded_filter_epochs_results_dict.items()}


## Plot ripples in `PhoPaginatedMultiDecoderDecodedEpochsWindow`

Pipeline/`track_templates` are unavailable from HDF5 alone, so build each decoder's `DecodedEpochSlicesPaginatedFigureController` from the reassembled ripple `DecodedFilterEpochsResult`s (synthetic `xbin`, skip measured positions), then combine with `init_from_pagination_controller_dict`.

In [ ]:
from neuropy.utils.result_context import IdentifyingContext
from pyphoplacecellanalysis.Pho2D.stacked_epoch_slices import PhoPaginatedMultiDecoderDecodedEpochsWindow, DecodedEpochSlicesPaginatedFigureController

## INPUTS: restored_decoded_filter_epochs_results_dict
decoder_ripple_filter_epochs_decoder_result_dict: Dict[str, DecodedFilterEpochsResult] = {decoder_name: epochs_dict['ripple'] for decoder_name, epochs_dict in restored_decoded_filter_epochs_results_dict.items() if 'ripple' in epochs_dict}
assert len(decoder_ripple_filter_epochs_decoder_result_dict) > 0, "no 'ripple' DecodedFilterEpochsResult entries to plot"
decoder_names: List[str] = list(decoder_ripple_filter_epochs_decoder_result_dict.keys())
print(f'ripple decoders: {decoder_names}')


## Merge exported score + shuffle CSVs onto ripple `filter_epochs`

Corner overlays read unsuffixed columns on each decoder's `filter_epochs` (`wcorr`, `score`, `mseq_tcov`, …). Shuffle z/p are not in the overlay allow-list by default — we temporarily extend `OverlayLabelsPlotData.HEURISTIC_LABEL_KEYS` so `wcorr_z` / `best_overall_wcorr_z` can render after merge.

In [ ]:
## INPUTS: active_posterior_h5_path, decoded_posteriors_parsed_h5_files_df, matching_custom_replay_name_str, decoding_time_bin_size_str, decoder_ripple_filter_epochs_decoder_result_dict

active_h5_meta_row = decoded_posteriors_parsed_h5_files_df[decoded_posteriors_parsed_h5_files_df['path'].astype(str).map(lambda p: Path(p).resolve() == active_posterior_h5_path)].iloc[0]
active_session_str: str = str(active_h5_meta_row['session'])
active_custom_replay_name: str = str(active_h5_meta_row.get('custom_replay_name', matching_custom_replay_name_str))
active_tbin_str: str = str(active_h5_meta_row.get('decoding_time_bin_size_str', decoding_time_bin_size_str))
print(f'active_session_str={active_session_str!r}, replay={active_custom_replay_name!r}, tbin={active_tbin_str!r}')

csv_files = find_csv_files(collected_outputs_directory)
_csv_sessions, most_recent_parsed_csv_files_df, _all_parsed_csv_files_df = find_most_recent_files(found_session_export_paths=csv_files, cuttoff_date=cuttoff_date)

def _subfn_pick_most_recent_csv(file_type: str) -> Optional[Path]:
    a_df = most_recent_parsed_csv_files_df[(most_recent_parsed_csv_files_df['session'].astype(str) == active_session_str) & (most_recent_parsed_csv_files_df['file_type'] == file_type)].copy()
    if len(a_df) == 0:
        return None
    if 'decoding_time_bin_size_str' in a_df.columns:
        tbin_match = a_df['decoding_time_bin_size_str'].astype(str).isin([active_tbin_str, f'{float(active_tbin_str):g}', f'{float(active_tbin_str):.3f}', f'{float(active_tbin_str):.4f}'])
        if bool(tbin_match.any()):
            a_df = a_df[tbin_match]
    if ('_comparable_custom_replay_name' in a_df.columns) and (matching_custom_replay_name_str is not None):
        cmp_match = a_df['_comparable_custom_replay_name'].astype(str) == str(matching_custom_replay_name_str)
        if bool(cmp_match.any()):
            a_df = a_df[cmp_match]
    elif 'custom_replay_name' in a_df.columns:
        replay_match = a_df['custom_replay_name'].astype(str) == active_custom_replay_name
        if bool(replay_match.any()):
            a_df = a_df[replay_match]
    a_df = a_df.sort_values('export_datetime', ascending=False)
    return Path(a_df.iloc[0]['path']).resolve()
## END def _subfn_pick_most_recent_csv...

scores_csv_path: Optional[Path] = _subfn_pick_most_recent_csv('ripple_all_scores_merged_df')
shuffle_csv_path: Optional[Path] = _subfn_pick_most_recent_csv('ripple_EpochScoreShuffle_df')
if shuffle_csv_path is None:
    shuffle_csv_path = _subfn_pick_most_recent_csv('ripple_WCorrShuffle_df') ## legacy fallback
print(f'scores_csv_path: {scores_csv_path}')
print(f'shuffle_csv_path: {shuffle_csv_path}')

active_session_all_scores_ripple_df: Optional[pd.DataFrame] = pd.read_csv(scores_csv_path) if (scores_csv_path is not None and scores_csv_path.exists()) else None
active_session_epoch_score_shuffle_df: Optional[pd.DataFrame] = pd.read_csv(shuffle_csv_path) if (shuffle_csv_path is not None and shuffle_csv_path.exists()) else None
if active_session_epoch_score_shuffle_df is not None:
    active_session_epoch_score_shuffle_df, epoch_score_shuffle_transfer_col_names = AcrossSessionHelpers.prepare_epoch_score_shuffle_transfer_columns(active_session_epoch_score_shuffle_df)
else:
    epoch_score_shuffle_transfer_col_names = []
print(f'scores rows: {None if active_session_all_scores_ripple_df is None else len(active_session_all_scores_ripple_df)}; shuffle transfer cols: {epoch_score_shuffle_transfer_col_names[:12]}{"…" if len(epoch_score_shuffle_transfer_col_names) > 12 else ""}')

## Base score / radon / heuristic names that appear as `{name}_{decoder}` on ripple_all_scores_merged_df:
SCORE_UNSUFFIX_BASE_NAMES: List[str] = ['P_decoder', 'score', 'velocity', 'intercept', 'speed', 'wcorr', 'pearsonr', 'travel', 'coverage', 'avg_jump_cm', 'max_jump', 'max_jump_cm', 'max_jump_cm_per_sec', 'ratio_jump_valid_bins', 'total_congruent_direction_change', 'longest_sequence_length', 'continuous_seq_sort', 'continuous_seq_len_ratio_no_repeats', 'main_contiguous_subsequence_len', 'mseq_len', 'mseq_len_ignoring_intrusions', 'mseq_len_ignoring_intrusions_and_repeats', 'mseq_len_ratio_ignoring_intrusions_and_repeats', 'mseq_tcov', 'mseq_dtrav', 'forwardicity']
SHUFFLE_OVERLAY_KEYS: List[str] = ['wcorr_z', 'best_overall_wcorr_z']

def _subfn_resolve_start_col(df: pd.DataFrame) -> str:
    for a_col in ('ripple_start_t', 'start', 'epoch_start_t'):
        if a_col in df.columns:
            return a_col
    raise KeyError(f'no start-time column in columns={list(df.columns)[:30]}')
## END def _subfn_resolve_start_col...

def _subfn_merge_score_and_shuffle_onto_decoder_result(a_result: DecodedFilterEpochsResult, decoder_name: str, scores_df: Optional[pd.DataFrame], shuffle_df: Optional[pd.DataFrame]) -> DecodedFilterEpochsResult:
    filter_epochs_df = ensure_dataframe(a_result.filter_epochs).copy()
    if 'start' not in filter_epochs_df.columns:
        raise KeyError(f'reassembled filter_epochs missing start for decoder={decoder_name}')
    filter_epochs_df['_join_start'] = pd.to_numeric(filter_epochs_df['start'], errors='coerce')

    if scores_df is not None and len(scores_df) > 0:
        suffix: str = f'_{decoder_name}'
        rename_map = {f'{base}{suffix}': base for base in SCORE_UNSUFFIX_BASE_NAMES if f'{base}{suffix}' in scores_df.columns}
        start_col = _subfn_resolve_start_col(scores_df)
        right = scores_df[[start_col, *list(rename_map.keys())]].copy() if len(rename_map) > 0 else scores_df[[start_col]].copy()
        right = right.rename(columns=rename_map)
        right['_join_start'] = pd.to_numeric(right[start_col], errors='coerce')
        right = right.drop(columns=[start_col]).drop_duplicates(subset=['_join_start'], keep='first')
        n_before = len(filter_epochs_df)
        filter_epochs_df = filter_epochs_df.merge(right, how='left', on='_join_start')
        assert len(filter_epochs_df) == n_before, f'scores merge changed row count for {decoder_name}'
        print(f'  {decoder_name}: merged {len(rename_map)} score cols; non-null wcorr={filter_epochs_df["wcorr"].notna().sum() if "wcorr" in filter_epochs_df.columns else 0}/{n_before}')

    if shuffle_df is not None and len(shuffle_df) > 0:
        start_col = _subfn_resolve_start_col(shuffle_df)
        per_decoder_z_col = f'wcorr_z_{decoder_name}'
        long_short_z_col = 'wcorr_z_long' if decoder_name.startswith('long') else ('wcorr_z_short' if decoder_name.startswith('short') else None)
        transfer_cols = []
        if per_decoder_z_col in shuffle_df.columns:
            transfer_cols.append(per_decoder_z_col)
        elif (long_short_z_col is not None) and (long_short_z_col in shuffle_df.columns):
            transfer_cols.append(long_short_z_col)
        for a_col in ('best_overall_wcorr_z', 'wcorr_z_long', 'wcorr_z_short'):
            if (a_col in shuffle_df.columns) and (a_col not in transfer_cols):
                transfer_cols.append(a_col)
        ## END for a_col in ('best_overall_wcorr_z', 'wcorr_z_long', 'wcorr_z_short')...

        right = shuffle_df[[start_col, *transfer_cols]].copy()
        right['_join_start'] = pd.to_numeric(right[start_col], errors='coerce')
        right = right.drop(columns=[start_col]).drop_duplicates(subset=['_join_start'], keep='first')
        ## Prefer decoder-specific z as overlay key `wcorr_z`:
        if per_decoder_z_col in right.columns:
            right = right.rename(columns={per_decoder_z_col: 'wcorr_z'})
        elif (long_short_z_col is not None) and (long_short_z_col in right.columns) and ('wcorr_z' not in right.columns):
            right = right.rename(columns={long_short_z_col: 'wcorr_z'})
        n_before = len(filter_epochs_df)
        filter_epochs_df = filter_epochs_df.merge(right, how='left', on='_join_start')
        assert len(filter_epochs_df) == n_before, f'shuffle merge changed row count for {decoder_name}'
        print(f'  {decoder_name}: merged shuffle cols {[c for c in right.columns if c != "_join_start"]}; non-null wcorr_z={filter_epochs_df["wcorr_z"].notna().sum() if "wcorr_z" in filter_epochs_df.columns else 0}/{n_before}')

    filter_epochs_df = filter_epochs_df.drop(columns=['_join_start'], errors='ignore')
    a_result.filter_epochs = filter_epochs_df
    return a_result
## END def _subfn_merge_score_and_shuffle_onto_decoder_result...

for decoder_name in list(decoder_ripple_filter_epochs_decoder_result_dict.keys()):
    decoder_ripple_filter_epochs_decoder_result_dict[decoder_name] = _subfn_merge_score_and_shuffle_onto_decoder_result(decoder_ripple_filter_epochs_decoder_result_dict[decoder_name], decoder_name=decoder_name, scores_df=active_session_all_scores_ripple_df, shuffle_df=active_session_epoch_score_shuffle_df)
## END for decoder_name in list(decoder_ripple_filter_epochs_decoder_result_dict.keys())...

## OUTPUTS: decoder_ripple_filter_epochs_decoder_result_dict (score+shuffle enriched), active_session_all_scores_ripple_df, active_session_epoch_score_shuffle_df, SHUFFLE_OVERLAY_KEYS


In [ ]:

## Synthetic position bins from posterior spatial axis (real decoder.xbin not stored in PosteriorExporting layout):
def _subfn_infer_xbin_from_result(a_result: DecodedFilterEpochsResult) -> np.ndarray:
    if a_result.pos_bin_edges is not None:
        return np.asarray(a_result.pos_bin_edges)
    first_p = np.asarray(a_result.p_x_given_n_list[0])
    n_xbins: int = int(first_p.shape[0])
    return np.arange(n_xbins + 1, dtype=float)
## END def _subfn_infer_xbin_from_result...

## Dummy measured-position DF (skipped via params; column still required by some helpers):
global_pos_df = pd.DataFrame({'t': np.asarray([], dtype=float), 'lin_pos': np.asarray([], dtype=float)})

max_subplots_per_page: int = 8
controlling_decoder_name: str = decoder_names[0]
pagination_controller_dict = {}
for decoder_name in decoder_names:
    a_result: DecodedFilterEpochsResult = decoder_ripple_filter_epochs_decoder_result_dict[decoder_name]
    is_controlling_widget: bool = (decoder_name == controlling_decoder_name)
    curr_params_kwargs = dict(skip_plotting_measured_positions=True, skip_plotting_most_likely_positions=True, isPaginatorControlWidgetBackedMode=True, epochs_name='ripple', is_controlled_widget=(not is_controlling_widget), disable_y_label=(not is_controlling_widget), enable_radon_transform_info=False, enable_overlay_labels_info=False, enable_update_window_title_on_page_change=False, build_internal_callbacks=False)
    pagination_controller_dict[decoder_name] = DecodedEpochSlicesPaginatedFigureController.init_from_decoder_data(a_result.filter_epochs, filter_epochs_decoder_result=a_result, xbin=_subfn_infer_xbin_from_result(a_result), global_pos_df=global_pos_df, a_name=f'DecodedEpochSlices[{decoder_name}]', active_context=IdentifyingContext(display_fn_name='DecodedEpochSlices', epochs='ripple', decoder=decoder_name), max_subplots_per_page=max_subplots_per_page, params_kwargs=curr_params_kwargs)
## END for decoder_name in decoder_names...


app, paginated_multi_decoder_decoded_epochs_window = PhoPaginatedMultiDecoderDecodedEpochsWindow.init_from_pagination_controller_dict(pagination_controller_dict, name='HDF5RecoveredRipplesWindow', title='HDF5-recovered Ripple Decoded Epochs', defer_show=False)
## OUTPUTS: app, paginated_multi_decoder_decoded_epochs_window, pagination_controller_dict, decoder_ripple_filter_epochs_decoder_result_dict
paginated_multi_decoder_decoded_epochs_window


## Add score / shuffle overlays (ReviewOfWork-style)

Same `add_data_overlays(included_columns=[...])` path as the interactive notebook. Shuffle z keys are temporarily allow-listed + formatted so they can appear in the green heuristic label block.

In [ ]:
## Temporarily allow shuffle z keys through OverlayLabelsPlotData (reverts if you restart kernel / reimport):
for k in SHUFFLE_OVERLAY_KEYS:
    if k not in OverlayLabelsPlotData.HEURISTIC_LABEL_KEYS:
        OverlayLabelsPlotData.HEURISTIC_LABEL_KEYS.append(k)
## END for k in SHUFFLE_OVERLAY_KEYS...

_orig_init_from_df_row = OverlayLabelsPlotData.init_from_df_row_tuple_and_formatting_fn_dict.__func__

@classmethod
def _patched_init_from_df_row_tuple_and_formatting_fn_dict(cls, a_tuple, column_formatting_fn_dict):
    column_formatting_fn_dict = dict(column_formatting_fn_dict)
    default_float_formatting_fn = lambda v: str(np.array([v])).lstrip("[").rstrip("]")
    for k in SHUFFLE_OVERLAY_KEYS:
        if column_formatting_fn_dict.get(k, None) is None:
            column_formatting_fn_dict[k] = (lambda v, name=k: f"{name}: {default_float_formatting_fn(v)}")
    ## END for k in SHUFFLE_OVERLAY_KEYS...
    return _orig_init_from_df_row(cls, a_tuple, column_formatting_fn_dict)
## END def _patched_init_from_df_row_tuple_and_formatting_fn_dict...

OverlayLabelsPlotData.init_from_df_row_tuple_and_formatting_fn_dict = _patched_init_from_df_row_tuple_and_formatting_fn_dict

# overlay_included_columns: List[str] = ['radon', 'wcorr', 'mseq_tcov', 'forwardicity', *SHUFFLE_OVERLAY_KEYS]
overlay_included_columns: List[str] = ['wcorr', 'mseq_tcov', 'forwardicity', *SHUFFLE_OVERLAY_KEYS]
print(f'overlay_included_columns: {overlay_included_columns}')

## Ensure child controllers see the enriched filter_epochs (window property reads plots_data.filter_epochs_decoder_result):
for decoder_name, a_ctrl in pagination_controller_dict.items():
    a_ctrl.plots_data.filter_epochs_decoder_result = decoder_ripple_filter_epochs_decoder_result_dict[decoder_name]
## END for decoder_name, a_ctrl in pagination_controller_dict.items()...

_out_add_data_overlays = paginated_multi_decoder_decoded_epochs_window.add_data_overlays(included_columns=overlay_included_columns)
## OUTPUTS: _out_add_data_overlays
_out_add_data_overlays


In [ ]:
## Optional: recover all filtered files into a session-keyed dict
# all_sessions_restored_single_epoch_dict: Dict[str, Dict[str, Dict[str, Dict[str, SingleEpochDecodedResult]]]] = {}
# for a_path in decoded_posteriors_h5_files:
#     restored_dict, _save_key = recover_single_epoch_decoded_results_from_hdf5(a_path, debug_print=False)
#     all_sessions_restored_single_epoch_dict[a_path.name] = restored_dict
#     print(f'{a_path.name}: { {d: {e: len(ep) for e, ep in ed.items()} for d, ed in restored_dict.items()} }')
# ## END for a_path in decoded_posteriors_h5_files...
# ## OUTPUTS: all_sessions_restored_single_epoch_dict
